## 📋 **Scenario: Gestione Giacimento - Perforazione, Assay e Geotecnica**
### Un progetto di esplorazione ha raccolto dati da **sondaggi**, **analisi geochimiche** e **test geotecnici** per valutare la fattibilità di una miniera a cielo aperto.
    
## 🎯 **QUESITI (con JOIN e CTE)**

### **A. Pulizia Dati**
**1. Crea 3 tabelle temporanee** con:
   - ID uniformi (uppercase, formato standard)
   - Date in `YYYY-MM-DD`
   - Valori numerici (gestisci `zero`, `ND`, spazi)
   - Testi in uppercase e trim

### **B. Analisi con JOIN e CTE**

**1. Report completo sondaggi**  
Per ogni sondaggio, mostra:
- `ID_Sondaggio`, `Sito`, `Profondità_m`, `Metodo`
- `num_campioni` (da ASSAY)
- `num_test` (da GEOTECNICA)
- `media_Au_ppm`
- `media_RQD`

**2. Classifica siti per qualità roccia**  
Calcola per ogni sito:
- `media_RQD`
- `media_Au_ppm`
- `classifica` (usando CTE: RQD > 75 = "Buona", 50-75 = "Media", < 50 = "Scarsa")

**3. Anomalie da investigare**  
Trova sondaggi con:
- `RQD < 50` MA `Au_ppm > 1.0` (roccia scadente ma mineralizzata)

**4. Profondità ottimale**  
Usando CTE, calcola per ogni sito:
- `profondità_media_Au_max` (profondità dove Au è massimo)
- `RQD_corrispondente`

**5. Efficienza perforazione**  
Per ogni `Metodo`, calcola:
- `media_profondità`
- `media_Au_ppm`
- `media_RQD`
- `num_sondaggi`

---
    
## ✅ **Obiettivi**

1. **Pulisci** le 3 tabelle (crea tabelle temporanee)
2. **JOIN** per rispondere ai 5 quesiti
3. **CTE** almeno in uno dei quesiti (suggerito quesiti 2, 3, 4)
4. **Scrivi tu le considerazioni finali** basate sui risultati

In [17]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Tabella 1: SONDAGGI
sondaggi = {
    'ID_Sondaggio': ['DH-101', 'dh-101', 'DH102 ', 'DH-103', 'DH-104'],
    'Data': ['2024-01-15', '20/02/2024', '2024.03.10', '2024-04-05', '15/05/2024'],
    'Sito': ['Nord', 'nord', 'SUD', 'sud', 'Est'],
    'Profondità_m': ['150', '200.5', 'zero', '175', '250'],
    'Metodo': ['RC', 'rc', 'Diamond', 'DIAMOND', 'RC']
}

# Tabella 2: ASSAY
assay = {
    'ID_Campione': ['S-101', 's-101', 'S102', 'S103 ', 'S104', 'S105'],
    'ID_Sondaggio': ['DH-101', 'DH101', 'DH-102', 'DH-102', 'DH-103', 'DH-104'],
    'Da_m': ['0.0', '10.5', '15', '20', '25', '30'],
    'A_m': ['1.0', '11.5', '16', '25', '30', '35'],
    'Au_ppm': ['0.3', '1.5', 'ND', '2.1', '0.8', '1.2'],
    'Cu_ppm': ['120', '250', 'ND', '85', '150', '200']
}

# Tabella 3: GEOTECNICA
geotecnica = {
    'ID_Test': ['GT-01', 'gt-01', 'GT02', 'GT-03', 'GT-04'],
    'ID_Sondaggio': ['DH-101', 'DH101', 'DH-102', 'DH-103', 'DH-104'],
    'Profondità_m': ['25', '50.5', '75', 'zero', '100'],
    'RQD': ['75', '85', 'ND', '92', '65'],
    'Tipo_Roccia': ['Granito', 'granito', 'SCISTO', 'Marmo', 'Granito']
}

# Carica in SQLite
df_sondaggi = pd.DataFrame(sondaggi)
df_assay = pd.DataFrame(assay)
df_geotecnica = pd.DataFrame(geotecnica)

df_sondaggi.to_sql('sondaggi', conn, index=False, if_exists='replace')
df_assay.to_sql('assay', conn, index=False, if_exists='replace')
df_geotecnica.to_sql('geotecnica', conn, index=False, if_exists='replace')

print("=== DATI CARICATI ===")
print("\nSONDAGGI:")
print(df_sondaggi)
print("\nASSAY:")
print(df_assay)
print("\nGEOTECNICA:")
print(df_geotecnica)

=== DATI CARICATI ===

SONDAGGI:
  ID_Sondaggio        Data  Sito Profondità_m   Metodo
0       DH-101  2024-01-15  Nord          150       RC
1       dh-101  20/02/2024  nord        200.5       rc
2       DH102   2024.03.10   SUD         zero  Diamond
3       DH-103  2024-04-05   sud          175  DIAMOND
4       DH-104  15/05/2024   Est          250       RC

ASSAY:
  ID_Campione ID_Sondaggio  Da_m   A_m Au_ppm Cu_ppm
0       S-101       DH-101   0.0   1.0    0.3    120
1       s-101        DH101  10.5  11.5    1.5    250
2        S102       DH-102    15    16     ND     ND
3       S103        DH-102    20    25    2.1     85
4        S104       DH-103    25    30    0.8    150
5        S105       DH-104    30    35    1.2    200

GEOTECNICA:
  ID_Test ID_Sondaggio Profondità_m RQD Tipo_Roccia
0   GT-01       DH-101           25  75     Granito
1   gt-01        DH101         50.5  85     granito
2    GT02       DH-102           75  ND      SCISTO
3   GT-03       DH-103         zero  

In [18]:
q_pulizia_sondaggi = """
SELECT
 
 -- fix ID_Sondaggio
CASE
WHEN UPPER(TRIM(ID_Sondaggio)) GLOB 'DH[0-9]*' THEN SUBSTR(ID_Sondaggio,1,2) || '-' || SUBSTR(ID_Sondaggio,3,3)
ELSE UPPER(TRIM(ID_Sondaggio))
END AS id_sondaggio,

-- fix Data
CASE 
WHEN Data LIKE '__/__/____' THEN SUBSTR(Data,7,4) || '-' || SUBSTR(Data,4,2) || '-' || SUBSTR(Data,1,2)
WHEN Data LIKE '%.%' THEN REPLACE(Data,'.','-')
ELSE TRIM(Data)
END AS data,

-- fix Sito
UPPER(TRIM(Sito)) AS sito,

-- fix Profondita'_m
CASE 
WHEN "Profondità_m" = 'zero' THEN 0.0
ELSE CAST(TRIM("Profondità_m") AS REAL) 
END AS profondita_m,

-- fix Metodo
 UPPER(TRIM(Metodo)) AS metodo

FROM sondaggi;
"""

cursor.execute("CREATE TEMPORARY TABLE sondaggi_puliti_temp AS" + q_pulizia_sondaggi)
print("=== SONDAGGI Table pulita ===")
df_verifica_sondaggi = pd.read_sql_query("SELECT * FROM sondaggi_puliti_temp", conn)
print(df_verifica_sondaggi)
print("\n\n")

=== SONDAGGI Table pulita ===
  id_sondaggio        data  sito  profondita_m   metodo
0       DH-101  2024-01-15  NORD         150.0       RC
1       DH-101  2024-02-20  NORD         200.5       RC
2       DH-102  2024-03-10   SUD           0.0  DIAMOND
3       DH-103  2024-04-05   SUD         175.0  DIAMOND
4       DH-104  2024-05-15   EST         250.0       RC



